In [ ]:
import sys
import pandas as pd
import numpy as np
from glob import glob
from sklearn.metrics import accuracy_score, f1_score, classification_report

sys.path.append("../Config")
from config import DATASETS, INPUT_SETTINGS, LABEL_SETTINGS

In [ ]:
# 100 random subsamples of 500 test memes (without replacement).
# Note: 500 memes is 1/6 of FHM's test set but half of MAMI's, so the stds are not comparable across datasets;
# set REPLACE = True and SAMPLE_SIZE = None for a standard bootstrap (n memes with replacement) on both datasets.
SAMPLE_SIZE = 500
REPLACE = False

def bootstrap_performance(y_true, y_pred, sample_size=500, n_samples=100, replace=False):
    assert len(y_true) == len(y_pred), "Both lists must be of same length"
    sample_size = len(y_true) if sample_size is None else min(sample_size, len(y_true))
    accuracies = []
    f1s = []

    for seed_val in range(n_samples):
        # Sample indices
        np.random.seed(seed_val)
        indices = np.random.choice(len(y_true), size=sample_size, replace=replace)
        sample_true = [y_true[i] for i in indices]
        sample_pred = [y_pred[i] for i in indices]
        acc = accuracy_score(sample_true, sample_pred)
        accuracies.append(acc)
        f1 = f1_score(sample_true, sample_pred, average='weighted')
        f1s.append(f1)

    return np.mean(accuracies), np.std(accuracies), np.mean(f1s), np.std(f1s)

def analyze_results(tru, pred):
    accuracy = accuracy_score(tru, pred)
    f1 = f1_score(tru, pred, average='weighted')
    report = classification_report(tru, pred, digits=6, zero_division=0)

    print(f"Overall Accuracy: {100*accuracy:.2f}")
    print(f"Overall w-F1 Score: {100*f1:.2f}")
    print("Overall Classification Report:")
    print(report)

    # Mean and standard deviation of the predictions performance scores
    mean_acc, std_acc, mean_f1, std_f1 = bootstrap_performance(tru, pred, SAMPLE_SIZE, replace=REPLACE)
    print(f"Bootstrapped Accuracy: {100*mean_acc:.2f} ± {100*std_acc:.2f}")
    print(f"Bootstrapped w-F1 Score: {100*mean_f1:.2f} ± {100*std_f1:.2f}")
    return mean_acc, std_acc, mean_f1, std_f1

In [ ]:
# (category, display name, size) of each model, in table order
MODELS = {
    "albert-base-v2": ("Unimodal (text)", "ALBERT", "12M"),
    "bert-base-uncased": ("Unimodal (text)", "BERT", "112M"),
    "roberta-base": ("Unimodal (text)", "RoBERTa", "125M"),
    "vit-base-patch16-224": ("Unimodal (vision)", "ViT", "86M"),
    "beit-base-patch16-224-pt22k-ft22k": ("Unimodal (vision)", "BEiT", "86M"),
    "deit-base-patch16-224": ("Unimodal (vision)", "DeiT", "86M"),
    "clip-vit-base-patch32": ("Multimodal", "CLIP", "150M"),
    "SmolVLM-Instruct": ("Multimodal", "SmolVLM", "2B"),
    "Qwen2.5-VL-3B-Instruct": ("Multimodal", "Qwen2.5-VL", "3B"),
    "Qwen2-VL-7B-Instruct": ("Multimodal", "Qwen2-VL", "7B"),
}
TEXT_MODELS = ["albert-base-v2", "bert-base-uncased", "roberta-base"]
VLMS = ["SmolVLM-Instruct", "Qwen2.5-VL-3B-Instruct", "Qwen2-VL-7B-Instruct"]

# Outputs/<dataset>/<model>_<setting>.csv (fine-tuned) or <model>_ZS_<setting>.csv (zero-shot)
def parse_file(f):
    stem = f.split("/")[-1][:-4]
    for setting in INPUT_SETTINGS + LABEL_SETTINGS:
        if stem.endswith("_" + setting):
            model = stem[: -len(setting) - 1]
            if model.endswith("_ZS"):
                return model[:-3], "ZS", setting
            return model, "FT", setting
    return None

In [ ]:
rows = []
for dataset in DATASETS:
    files = glob(f"Outputs/{dataset}/*.csv")
    files.sort()

    for f in files:
        parsed = parse_file(f)
        if parsed is None:
            continue
        model, learning, setting = parsed
        print(f"Analyzing file: {f}")
        df = pd.read_csv(f)
        tru = df['true_label'].tolist()
        pred = df['predicted_label'].tolist()
        if (df['predicted_label'] == 2).any():
            print(f"Unclear outputs (counted as wrong): {(df['predicted_label'] == 2).sum()}")

        # Analyze the results
        mean_acc, std_acc, mean_f1, std_f1 = analyze_results(tru, pred)
        rows.append({"dataset": dataset, "model": model, "learning": learning, "setting": setting,
                     "acc": 100*mean_acc, "acc_std": 100*std_acc, "f1": 100*mean_f1, "f1_std": 100*std_f1})
        print("\n" + "="*50 + "\n")

results = pd.DataFrame(rows)
results.to_csv("Outputs/results_all.csv", index=False)

### Result tables
Cells are `mean ± std` over the bootstrap samples; `-` means the run is missing. Tables are also saved to `Outputs/tables/`.

In [ ]:
def cell(dataset, model, learning, setting, metric):
    r = results[(results.dataset == dataset) & (results.model == model) & (results.learning == learning) & (results.setting == setting)] if len(results) else results
    if len(r) == 0:
        return "-"
    return f"{r.iloc[0][metric]:.2f} ± {r.iloc[0][metric + '_std']:.2f}"

# Labels: prompt (simple/category) x label (binary/scaled), fine-tuning and zero-shot prompting
def labels_table(dataset):
    out = []
    for learning, models in [("FT", TEXT_MODELS + VLMS), ("ZS", VLMS)]:
        for model in models:
            category, name, size = MODELS[model]
            for prompt in ["simple", "category"]:
                row = {"Category": f"{category} ({'fine-tuning' if learning == 'FT' else 'prompting'})", "Model": name, "Size": size, "Prompt": prompt.title()}
                for col, label in [("Original (Binary)", "binary"), ("Augmented (Scaled)", "scale")]:
                    row[f"{col} Accuracy"] = cell(dataset, model, learning, f"{prompt}_{label}", "acc")
                    row[f"{col} F1-score"] = cell(dataset, model, learning, f"{prompt}_{label}", "f1")
                out.append(row)
    return pd.DataFrame(out)

# Inputs: original vs extended training data
def inputs_table(dataset):
    out = []
    for model, (category, name, size) in MODELS.items():
        row = {"Category": f"{category} (fine-tuning)", "Model": name, "Size": size}
        for col, setting in [("Original", "orig"), ("Augmented (Extended)", "inputs")]:
            row[f"{col} Accuracy"] = cell(dataset, model, "FT", setting, "acc")
            row[f"{col} F1-score"] = cell(dataset, model, "FT", setting, "f1")
        out.append(row)
    return pd.DataFrame(out)

# Zero-shot VLMs with the Yes/No prompt
def zeroshot_table(dataset):
    out = []
    for model in VLMS:
        category, name, size = MODELS[model]
        out.append({"Category": f"{category} (zero-shot ICL)", "Model": name, "Size": size,
                    "Accuracy": cell(dataset, model, "ZS", "orig", "acc"), "F1-score": cell(dataset, model, "ZS", "orig", "f1")})
    return pd.DataFrame(out)

In [ ]:
import os
os.makedirs("Outputs/tables", exist_ok=True)

for dataset in DATASETS:
    for name, table in [("labels", labels_table), ("inputs", inputs_table), ("zeroshot", zeroshot_table)]:
        t = table(dataset)
        t.to_csv(f"Outputs/tables/{dataset}_{name}.csv", index=False)
        print(f"##### {dataset}: {name} #####")
        display(t)